### 1. Load libraries

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

from transformers import (
    AutoConfig,
    AutoModel,
    Trainer,
    TrainingArguments,
    EvalPrediction,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model
from safetensors.torch import load_file
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from dataset import cls_weights, train_ds, test_ds, val_ds, BEAT_WINDOW, CLASS_NAMES, y_test, DEVICE, val_ds, y_val
import matplotlib.pyplot as plt

### 2. Set hyperparameters

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
N_CLASSES = 5
DROPOUT = 0.15

PATCH_SIZE = 10
PATCH_STRIDE = 10
USE_LORA = True

BATCH_SIZE = 128
EPOCHS = 30
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 1e-2
MAX_GRAD_NORM = 1.0

### 3. Define the model

In [ ]:
def ecg_collator(features):
    x = torch.stack([f["input_values"] for f in features], dim=0)
    y = torch.stack([f["labels"] for f in features], dim=0)
    return {
        "input_values": x,
        "labels": y
    }


class ECGPatchEmbed(nn.Module):
    """
    Convert a beat of length 250 into patch embeddings.
    Input:  (B, 250)
    Output: (B, T, hidden_size)
    """
    def __init__(self, seq_len: int, patch_size: int, stride: int, hidden_size: int):
        super().__init__()
        self.seq_len = seq_len
        self.patch_size = patch_size
        self.stride = stride
        self.unfold = nn.Unfold(kernel_size=(1, patch_size), stride=(1, stride))
        self.proj = nn.Linear(patch_size, hidden_size)

        num_patches = ((seq_len - patch_size) // stride) + 1
        self.num_patches = num_patches

    def forward(self, x):
        x = x.unsqueeze(1).unsqueeze(2)
        patches = self.unfold(x)
        patches = patches.transpose(1, 2)
        return self.proj(patches)


class QwenECGClassifier(nn.Module):
    def __init__(self, model_name: str, n_classes: int, use_lora=True):
        super().__init__()

        # load pretrained model via HF
        self.config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
        self.backbone = AutoModel.from_pretrained(
            model_name,
            trust_remote_code=True,
            torch_dtype=torch.float32,
        )
        print(self.config, self.backbone)

        text_cfg = getattr(self.config, "text_config", self.config)
        hidden_size = text_cfg.hidden_size
        self.hidden_size = hidden_size

        # LORA config
        if use_lora:
            lora_config = LoraConfig(
                r=8,
                lora_alpha=16,
                lora_dropout=0.1,
                bias="none",
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
                task_type="FEATURE_EXTRACTION",
            )
            self.backbone = get_peft_model(self.backbone, lora_config)
            self.backbone.print_trainable_parameters()
        else:
            for p in self.backbone.parameters():
                p.requires_grad = False

        # patch embeddings
        self.patch_embed = ECGPatchEmbed(
            seq_len=BEAT_WINDOW,
            patch_size=PATCH_SIZE,
            stride=PATCH_STRIDE,
            hidden_size=hidden_size
        )

        # trainable classification token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, hidden_size))

        # positional embeddings
        self.pos_embed = nn.Parameter(
            torch.zeros(1, self.patch_embed.num_patches + 1, hidden_size)
        )

        # classifier part
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(DROPOUT)
        self.classifier = nn.Linear(hidden_size, n_classes)

        nn.init.normal_(self.cls_token, std=0.02)
        nn.init.normal_(self.pos_embed, std=0.02)

    def forward(self, input_values=None, labels=None):
        B = input_values.size(0)

        x = self.patch_embed(input_values)
        cls_tok = self.cls_token.expand(B, -1, -1)
        x = torch.cat([x, cls_tok], dim=1)
        x = x + self.pos_embed[:, :x.size(1), :]

        attention_mask = torch.ones(
            x.size(0), x.size(1),
            dtype=torch.long,
            device=x.device
        )

        outputs = self.backbone(
            inputs_embeds=x,
            attention_mask=attention_mask,
            output_hidden_states=False,
            return_dict=True
        )

        # last token pooling
        pooled = outputs.last_hidden_state[:, -1]
        pooled = self.norm(pooled)
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)

        return {"loss": None, "logits": logits}


class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs, labels=labels)
        logits = outputs["logits"]

        loss_function = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_function(logits, labels)

        return (loss, outputs) if return_outputs else loss

    def create_optimizer_and_scheduler(self, num_training_steps: int):
        """Replace the default linear LR schedule with CosineAnnealingLR."""
        self.optimizer = AdamW(
            self.model.parameters(),
            lr=self.args.learning_rate,
            weight_decay=self.args.weight_decay,
        )
        self.lr_scheduler = CosineAnnealingLR(
            self.optimizer,
            T_max=num_training_steps,
            eta_min=5e-6,
        )


class MetricsCallback(TrainerCallback):
    def __init__(self):
        self.history = {
            "epoch": [],
            "train_loss": [],
            "val_loss": [],
            "val_accuracy": [],
            "val_f1_macro": [],
            "val_f1_weighted": [],
            "val_precision_macro": [],
            "val_recall_macro": [],
        }

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return

        # training loss logs
        if "loss" in logs and "epoch" in logs and "eval_loss" not in logs:
            self.history["train_loss"].append(logs["loss"])

        # eval logs
        if "eval_loss" in logs and "epoch" in logs:
            self.history["epoch"].append(logs["epoch"])
            self.history["val_loss"].append(logs.get("eval_loss"))
            self.history["val_accuracy"].append(logs.get("eval_accuracy"))
            self.history["val_f1_macro"].append(logs.get("eval_f1_macro"))
            self.history["val_f1_weighted"].append(logs.get("eval_f1_weighted"))
            self.history["val_precision_macro"].append(logs.get("eval_precision_macro"))
            self.history["val_recall_macro"].append(logs.get("eval_recall_macro"))


def compute_metrics(eval_pred: EvalPrediction):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, preds, average="macro", zero_division=0),
    }


model = QwenECGClassifier(
    model_name=MODEL_NAME,
    n_classes=N_CLASSES,
    use_lora=USE_LORA
)

model = model.to(DEVICE)

### 4. Set training arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen25_ecg_classifier",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    bf16=False,
    report_to="none",
    max_grad_norm=MAX_GRAD_NORM,
    remove_unused_columns=False,
)

metrics_callback = MetricsCallback()
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=ecg_collator,
    compute_metrics=compute_metrics,
    class_weights=cls_weights.to(DEVICE),
    callbacks=[metrics_callback],
)

### 5. Run the trainer

In [ ]:
#trainer.train()

### 6. Final evaluation

In [ ]:
hist = metrics_callback.history
epochs = hist["epoch"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, hist["val_accuracy"], marker="o", label="Val Accuracy", color='lightblue')
axes[0].plot(epochs, hist["val_f1_macro"], marker="o", label="Val Macro F1", color='darkblue')
axes[0].set_title("Validation Metrics")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Score")
#axes[0].set_xticks(epochs)
axes[0].grid(True, alpha=0.3)
axes[0].legend()

train_x = list(range(1, len(hist["train_loss"]) + 1))
axes[1].plot(train_x, hist["train_loss"], marker="o", label="Train Loss")
axes[1].plot(train_x, hist["val_loss"], marker="o", label="Val Loss")
axes[1].set_title("Train & Val Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
#axes[1].set_xticks(epochs)
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
best_ckpt = "/home/truepeak/PycharmProjects/thesis/notebooks/ecg_experiment/qwen25_ecg_classifier/checkpoint-4193"
model = QwenECGClassifier(
    model_name=MODEL_NAME,
    n_classes=N_CLASSES,
    use_lora=USE_LORA
).to(DEVICE)

state_dict = load_file(f"{best_ckpt}/model.safetensors")
model.load_state_dict(state_dict, strict=True)

trainer.model = model

test_output = trainer.predict(val_ds)
test_logits = test_output.predictions
test_preds = np.argmax(test_logits, axis=1)

print("\nTest classification report:")
print(classification_report(y_val, test_preds, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(y_val, test_preds)
row_sums = cm.sum(axis=1, keepdims=True)
cm_relative = np.divide(cm, row_sums, where=row_sums != 0)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_relative, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(cmap="viridis", ax=ax, colorbar=True, values_format=".2f")
ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
plt.tight_layout()
plt.show()